# Post-Correction – Qwen3 mit Few-Shot Beispielsätzen

**Unterschied zu `post_correction_all.ipynb`:**  
Die 20 manuell transkribierten Beispielsätze aus `data/few_shots_20_sentences.txt` werden  
in den System-Prompt eingebettet. Qwen sieht dadurch, wie korrekte Sätze im Cook/Forster-Datensatz  
aussehen sollen (Stil, Vokabular, archaisches Englisch, Interpunktion).

**Voraussetzungen:**
- Konfidenz-Dateien müssen bereits vorhanden sein (`data/confidence/`) → TrOCR-Stufe nicht nötig
- `data/few_shots_20_sentences.txt` mit 20 Beispielsätzen

**Ausgabe:** `data/corrected_transcriptions_fewshot/` (getrennt vom ursprünglichen Run)

In [1]:
import gc
import json
import re
import warnings
from pathlib import Path

import torch

warnings.filterwarnings('ignore')

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

MANIFEST_PATH   = REPO_ROOT / 'data' / 'all_manifests' / 'manifest.json'
CONF_DIR        = REPO_ROOT / 'data' / 'confidence'
TRANSCR_DIR     = REPO_ROOT / 'data' / 'transcriptions'
CORRECTED_DIR   = REPO_ROOT / 'data' / 'corrected_transcriptions_fewshot'
DEBUG_DIR       = REPO_ROOT / 'data' / 'debug_logs_fewshot'
CORR_DOC_PATH   = REPO_ROOT / 'data' / 'corrected_raw_document_fewshot.txt'
FEW_SHOT_PATH   = REPO_ROOT / 'data' / 'few_shots_20_sentences.txt'

# Ground-Truth Zeilen-Transkriptionen (manuell, human-verified) → Domänen-Lexikon
GT_LINE_DIRS = [
    REPO_ROOT / 'data' / 'train_set_line_crops',
    REPO_ROOT / 'data' / 'test_set_line_crops',
]

CORRECTED_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

CONF_THRESHOLD  = 0.75   # erhöht von 0.70 → weniger Wörter werden markiert
MAX_EDIT_RATIO  = 0.25   # Levenshtein-Filter: max. 25% der Wortlänge, min. 2

QWEN_MODEL_ID   = 'Qwen/Qwen3-4B'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device              : {device}')
print(f'Konfidenz-Schwelle  : {CONF_THRESHOLD}')
print(f'Max. Edit-Distanz   : max(2, len(wort) × {MAX_EDIT_RATIO})')
print(f'Ausgabe-Dir         : {CORRECTED_DIR}')
print(f'Few-Shot-Datei      : {FEW_SHOT_PATH}')

conf_files_count = len(list(CONF_DIR.glob('*.txt')))
corr_done        = len(list(CORRECTED_DIR.glob('*.txt')))
print(f'Konfidenz-Dateien   : {conf_files_count}')
print(f'Bereits korrigiert  : {corr_done}')

Device              : cuda
Konfidenz-Schwelle  : 0.75
Max. Edit-Distanz   : max(2, len(wort) × 0.25)
Ausgabe-Dir         : /home/justin/Ginger_Gradient/14/project/Capstone-Project/data/corrected_transcriptions_fewshot
Few-Shot-Datei      : /home/justin/Ginger_Gradient/14/project/Capstone-Project/data/few_shots_20_sentences.txt
Konfidenz-Dateien   : 923
Bereits korrigiert  : 0


In [2]:
with open(MANIFEST_PATH, encoding='utf-8') as f:
    global_manifest = json.load(f)

def sort_key(p):
    m = re.match(r'B(\d+)_P(\d+)', p['page_id'])
    return (int(m.group(1)), int(m.group(2))) if m else (99, 99)

pages = sorted(global_manifest['pages'], key=sort_key)
print(f'Seiten gesamt: {len(pages)}')

Seiten gesamt: 923


---
## Few-Shot Beispielsätze laden

Die 20 Sätze werden als Abschnitt in den System-Prompt eingebettet.  
Qwen lernt dadurch den Stil und das Vokabular des Datensatzes kennen.

In [3]:
raw = FEW_SHOT_PATH.read_text(encoding='utf-8').strip()

# Sätze extrahieren (Format: - "Satz")
few_shot_sentences = re.findall(r'- "(.+?)"', raw, re.DOTALL)
# Mehrzeilige Sätze normalisieren
few_shot_sentences = [' '.join(s.split()) for s in few_shot_sentences]

print(f'{len(few_shot_sentences)} Beispielsätze geladen:')
for i, s in enumerate(few_shot_sentences, 1):
    print(f'  {i:2d}. {s[:90]}...' if len(s) > 90 else f'  {i:2d}. {s}')

19 Beispielsätze geladen:
   1. I left London in the year 1767, came late in the Evening & in a very mysterious manner tol...
   2. I heard that Mr. Barrington's, letter had been read by Mr. Buller at the board & that Lord...
   3. The same country continues to Murrelgreen; where we went to Basingstock an ancient town; f...
   4. The bird being very fat & heavy it mask presents a great surface to the white in order to ...
   5. We saw several pieces of Ice some of which were of a singular appearance, something like t...
   6. This day at 7 o'clock every man on board, was reduced to a pint of water allowance & smoot...
   7. During night the wind-increased & at 4 0' Clock in the morning we passed for about 1/4 of ...
   8. We are now advancing very fast towards the meridian under which Mr. Bouvet pretended to ha...
   9. We saw about 4 whales; an Island Ice passed & on it was a large bird sitting, very nearly ...
  10. Several small pieces of Ice were observed, but by no means equal to 

---
## Domänen-Lexikon aus Ground-Truth aufbauen

Alle manuell transkribierten Zeilen (`train_set_line_crops/`, `test_set_line_crops/`)
plus die Few-Shot-Sätze bilden ein **human-verified Lexikon**.

Da die Wörter von Menschen verifiziert wurden, ist jedes Lexikon-Wort vertrauenswürdig —
auch bei nur einem Vorkommen. Das Lexikon wird später in Stufe 3 genutzt, um
TrOCR-Fehler zu korrigieren, die **über** dem Konfidenz-Threshold lagen
(z.B. `Nationalist(0.77)` → `Naturalist`).

In [4]:
from collections import Counter

def _word_core(w):
    """Entfernt führende/anhängende Interpunktion, behält Wortinneres (z.B. o'clock)."""
    return w.strip('.,;:!?"\'()[]&—–-')

LEXICON = Counter()

# 1) Ground-Truth Zeilen-Transkriptionen
n_gt_files = 0
for gt_dir in GT_LINE_DIRS:
    for p in gt_dir.rglob('*.txt'):
        n_gt_files += 1
        for w in p.read_text(encoding='utf-8').split():
            core = _word_core(w)
            if core:
                LEXICON[core] += 1

# 2) Few-Shot-Sätze (ebenfalls manuell transkribiert)
for s in few_shot_sentences:
    for w in s.split():
        core = _word_core(w)
        if core:
            LEXICON[core] += 1

# Lowercase-Lookup: Wort gilt als "bekannt" unabhängig von Groß-/Kleinschreibung
LEXICON_LOWER = {}
for w, c in LEXICON.items():
    lw = w.lower()
    # Bevorzugte Schreibweise = häufigste Variante
    if lw not in LEXICON_LOWER or c > LEXICON[LEXICON_LOWER[lw]]:
        LEXICON_LOWER[lw] = w

print(f'Ground-Truth-Dateien : {n_gt_files}')
print(f'Lexikon-Größe        : {len(LEXICON):,} Wortformen, {sum(LEXICON.values()):,} Tokens')
for w in ('Naturalist', 'Draughtsman', 'Resolution', 'Nationalist'):
    print(f'  {w:15s}: {LEXICON.get(w, 0)}×')

Ground-Truth-Dateien : 1109
Lexikon-Größe        : 2,485 Wortformen, 9,658 Tokens
  Naturalist     : 1×
  Draughtsman    : 1×
  Resolution     : 2×
  Nationalist    : 0×


---
## Qwen3-4B laden

In [5]:
import importlib
import subprocess
import sys

import ftfy
from transformers import AutoModelForCausalLM, AutoTokenizer

if importlib.util.find_spec('ftfy') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy'])

QWEN_FALLBACK_MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

def _load_fp16(model_id):
    return AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map='auto'
    )

def _load_4bit(model_id):
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    return AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map='auto'
    )

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)

try:
    qwen_model = _load_fp16(QWEN_MODEL_ID)
    load_mode  = 'float16'
except torch.cuda.OutOfMemoryError:
    print('float16 zu groß – versuche 4-bit...')
    if importlib.util.find_spec('bitsandbytes') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.46.1'])
    try:
        qwen_model = _load_4bit(QWEN_MODEL_ID)
        load_mode  = '4-bit NF4'
    except Exception as e:
        print(f'4-bit fehlgeschlagen ({e}) – Fallback auf {QWEN_FALLBACK_MODEL_ID}')
        QWEN_MODEL_ID  = QWEN_FALLBACK_MODEL_ID
        qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)
        qwen_model     = _load_fp16(QWEN_MODEL_ID)
        load_mode      = 'float16 (Fallback)'

qwen_model.eval()
print(f'Qwen geladen ({load_mode}): {QWEN_MODEL_ID}')
if torch.cuda.is_available():
    used_gb  = torch.cuda.memory_allocated(device) / 1024**3
    total_gb = torch.cuda.get_device_properties(device).total_memory / 1024**3
    print(f'VRAM: {used_gb:.1f} / {total_gb:.1f} GB')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen geladen (float16): Qwen/Qwen3-4B
VRAM: 7.5 / 11.6 GB


---
## Prompt-Aufbau mit Few-Shot Beispielen

Die 20 Beispielsätze werden als eigener Abschnitt in den System-Prompt eingefügt,  
direkt nach dem Kontext-Block und vor den eigentlichen Aufgabenanweisungen.

In [6]:
_IS_QWEN3 = 'Qwen3' in QWEN_MODEL_ID

_CONTEXT = """\
You are an expert in historical manuscript transcription.
Context: Johann Reinhold Forster's journal of Captain Cook's second voyage \
(HMS Resolution, 1772–1774).
Language: late 18th-century English and German, with nautical, botanical, \
and zoological terminology.
Topics: navigation, natural history (botany, zoology), encounters with Pacific \
indigenous peoples, shipboard life, Forster's relationship with Cook and the Admiralty.\
"""

# Few-shot block: Qwen sieht wie korrekte Sätze im Datensatz aussehen
_FEW_SHOT_BLOCK = "\nHere are 20 correctly transcribed sentences from this manuscript "\
                  "(for style and vocabulary reference):\n"
for _s in few_shot_sentences:
    _FEW_SHOT_BLOCK += f'  - "{_s}"\n'

STEP1_SYSTEM = _CONTEXT + _FEW_SHOT_BLOCK + """
You will receive an OCR-transcribed manuscript page.
Words with low OCR confidence are marked <<word>>.

Your task: identify ALL words that need correction:
  1. Every <<word>> that does not fit the context
  2. Any clearly nonsensical or misspelled UNMARKED word \
(garbled sequences, digits replacing letters, truncated words)

For each word to correct, output exactly one line:
  <<original>> → replacement  (reason: brief justification)

If a marked word already fits the context well, output:
  <<original>> → [keep]

Rules:
- The replacement MUST visually resemble the original (same length, same first and last \
letter, minimal character changes). If you cannot find a visually similar correction, \
output [keep] instead.
- CRITICAL — proper nouns, place names, person names, nouns, and verbs: it is ALWAYS \
better to leave the word misspelled than to replace it with a different word entirely. \
  BAD examples (forbidden substitutions):
    George → German   (different word, wrong meaning)
    Naturalist → Nationalist   (different word, changes meaning)
    Resolution → Revolution   (ship name must not be altered)
    Forster → Foster   (person name: preserve even if uncertain)
  GOOD examples (visually similar, same word with OCR fix):
    Natnralist → Naturalist   (n→u, clearly same word)
    Capt. → Capt.   ([keep], abbreviation fits context)
- Do NOT invent a word that looks nothing like the original
- Output ONLY the suggestion list — no introduction, no trailing text
"""

STEP2_SYSTEM = _CONTEXT + _FEW_SHOT_BLOCK + """
You will receive:
1. The OCR text with <<word>> markers for uncertain words
2. A correction suggestion list from a first analysis pass

Your task: produce the final corrected text.
- Apply all suggestions (ignore [keep] entries — leave those words unchanged and \
remove their <<>> markers)
- Remove ALL <<>> markers (keep the word as-is if no suggestion covers it)
- Do NOT change anything that is not listed in the suggestions
- Preserve line breaks exactly as in the input
- Preserve the page header (=== Book X, Page NNN ===) exactly
- Output ONLY the corrected text — no markers, no explanations, nothing else
"""

print('Prompts aufgebaut.')
print(f'  Kontext-Block      : {len(_CONTEXT)} Zeichen')
print(f'  Few-Shot-Block     : {len(_FEW_SHOT_BLOCK)} Zeichen ({len(few_shot_sentences)} Sätze)')
print(f'  STEP1_SYSTEM Länge : {len(STEP1_SYSTEM)} Zeichen')
print(f'  STEP2_SYSTEM Länge : {len(STEP2_SYSTEM)} Zeichen')

Prompts aufgebaut.
  Kontext-Block      : 430 Zeichen
  Few-Shot-Block     : 4503 Zeichen (19 Sätze)
  STEP1_SYSTEM Länge : 6420 Zeichen
  STEP2_SYSTEM Länge : 5547 Zeichen


---
## Hilfsfunktionen (2-Stufen-Pipeline)

In [7]:
def clean_encoding(text):
    return ftfy.fix_text(text)


def _join_hyphenated_lines(text):
    """
    Verbindet Wörter, die am Zeilenende mit Bindestrich getrennt wurden.
    'word-\\nnextword rest' → 'wordnextword rest'
    '<<word->>\\nnextword rest' → '<<wordnextword>> rest'
    """
    text = re.sub(r'<<([^>]+)->>(\n)(\S+)', r'<<\1\3>>', text)
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    return text


def _levenshtein(a, b):
    """Levenshtein edit distance (case-insensitive)."""
    a, b = a.lower(), b.lower()
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            temp = dp[j]
            dp[j] = prev if a[i-1] == b[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = temp
    return dp[n]


def _filter_suggestions(suggestions):
    """
    Filtert Step-1-Vorschläge anhand der Levenshtein-Distanz.
    Vorschläge mit dist > max(2, len(original) × MAX_EDIT_RATIO) → automatisch [keep].
    """
    filtered = []
    for line in suggestions.splitlines():
        m = re.match(r'<<([^>]+)>>\s*(?:→|->)\s*(\S+)', line)
        if not m:
            filtered.append(line)
            continue
        original, replacement = m.group(1), m.group(2)
        if replacement.startswith('[keep]'):
            filtered.append(line)
            continue
        repl_clean = re.sub(r'[^\w]', '', replacement)
        dist     = _levenshtein(original, repl_clean)
        max_dist = max(2, int(len(original) * MAX_EDIT_RATIO))
        if dist <= max_dist:
            filtered.append(line)
        else:
            filtered.append(
                f'<<{original}>> → [keep]  '
                f'(auto-rejected: edit_dist={dist} > max={max_dist})'
            )
    return '\n'.join(filtered)


def _apply_suggestions(annotated_text, suggestions_filtered):
    """
    Wendet gefilterte Vorschläge programmatisch an — kein zweiter Modell-Aufruf.
    """
    text = annotated_text
    n_rejected = 0

    for line in suggestions_filtered.splitlines():
        m = re.match(r'<<([^>]+)>>\s*(?:→|->)\s*(\S+)', line)
        if not m:
            continue
        original, replacement = m.group(1), m.group(2)
        marker = f'<<{original}>>'
        if replacement.startswith('[keep]') or 'auto-rejected' in line:
            text = text.replace(marker, original)
            if 'auto-rejected' in line:
                n_rejected += 1
        else:
            text = text.replace(marker, replacement)

    remaining = len(re.findall(r'<<[^>]+>>', text))
    text = re.sub(r'<<([^>]+)>>', r'\1', text)

    return text, n_rejected, remaining


def conf_file_to_annotated(conf_path):
    """
    Konfidenz-Datei → (annotated_text, n_low).
    Seitenköpfe werden von Deutsch nach Englisch übersetzt.
    Zeilenend-Bindestriche werden automatisch verbunden.
    """
    lines     = conf_path.read_text(encoding='utf-8').splitlines()
    out_lines = []
    n_low     = 0

    for line in lines:
        if line.startswith('SCHWELLE') or not line.strip():
            continue
        if line.startswith('UNSICHERE_WÖRTER:'):
            try:
                n_low = int(line.split(':')[1].strip().split()[0])
            except (IndexError, ValueError):
                pass
            continue
        if line.startswith('==='):
            line = re.sub(r'=== Buch (\d+), Seite (\d+) ===', r'=== Book \1, Page \2 ===', line)
            out_lines.append(line)
            continue
        parts     = line.split('] ', 1)
        word_part = parts[1] if len(parts) == 2 else line
        word_part = re.sub(r'<<([^>]+)>>\([\d.]+\)', r'<<\1>>', word_part)
        word_part = re.sub(r'(\S+)\([\d.]+\)', r'\1', word_part)
        out_lines.append(word_part.strip())

    text = '\n'.join(out_lines)
    text = _join_hyphenated_lines(text)
    return text, n_low


def _run_model(messages, enable_thinking, temperature, max_new_tokens):
    if _IS_QWEN3 and not enable_thinking:
        msgs = list(messages)
        msgs[-1] = {**msgs[-1], 'content': msgs[-1]['content'] + '\n/no_think'}
    else:
        msgs = messages

    prompt    = qwen_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs    = qwen_tokenizer(prompt, return_tensors='pt').to(qwen_model.device)
    do_sample = temperature > 0.05
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=qwen_tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs['temperature'] = temperature

    with torch.no_grad():
        output_ids = qwen_model.generate(**inputs, **gen_kwargs)

    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    result     = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Geschlossene UND ungeschlossene <think>-Blöcke entfernen
    # (ungeschlossen = Modell hat Token-Limit im Thinking erreicht → Output ist unbrauchbar)
    result     = re.sub(r'<think>.*?</think>', '', result, flags=re.DOTALL)
    result     = re.sub(r'<think>.*$', '', result, flags=re.DOTALL).strip()
    return result


def postprocess_with_qwen(annotated_text, page_id):
    text    = clean_encoding(annotated_text)
    n_words = len(text.split())

    # Step 1: Qwen schlägt Korrekturen vor.
    # WICHTIG: thinking DEAKTIVIERT — Qwen3-4B verliert sich sonst in endlosen
    # <think>-Schleifen und erreicht das Token-Limit ohne je Vorschläge auszugeben.
    step1_msgs = [
        {'role': 'system', 'content': STEP1_SYSTEM},
        {'role': 'user',   'content': text},
    ]
    suggestions_raw = _run_model(
        step1_msgs, enable_thinking=False,
        temperature=0.3, max_new_tokens=max(512, n_words * 3),
    )

    # Levenshtein-Filter: zu große Ersetzungen → [keep]
    suggestions_filtered = _filter_suggestions(suggestions_raw)

    # Step 2: programmatische Substitution — kein zweiter Modell-Aufruf
    corrected, n_rejected, n_remaining = _apply_suggestions(text, suggestions_filtered)

    m = re.match(r'B(\d+)_P(\d+)', page_id)
    if m and not corrected.startswith('==='):
        corrected = f'=== Book {m.group(1)}, Page {m.group(2)} ===\n' + corrected

    return corrected, suggestions_raw, suggestions_filtered, n_rejected, n_remaining


print('Hilfsfunktionen definiert.')
print('Step 1: thinking deaktiviert (verhindert Endlos-<think>-Schleifen)')
print('Step 2: programmatische Substitution (kein zweiter Modell-Aufruf)')

Hilfsfunktionen definiert.
Step 1: thinking deaktiviert (verhindert Endlos-<think>-Schleifen)
Step 2: programmatische Substitution (kein zweiter Modell-Aufruf)


---
## Korrektur ausführen

In [8]:
from tqdm import tqdm

n_corrected  = 0
n_copied     = 0
n_skipped    = 0
n_rejected   = 0   # vom Levenshtein-Filter abgelehnte Vorschläge
n_remaining  = 0   # Marker ohne Vorschlag (Wort unverändert gelassen)

for page_entry in tqdm(pages, desc='Qwen Few-Shot Postkorrektur'):
    page_id        = page_entry['page_id']
    conf_path      = CONF_DIR      / f'{page_id}.txt'
    corrected_path = CORRECTED_DIR / f'{page_id}.txt'
    orig_path      = TRANSCR_DIR   / f'{page_id}.txt'
    debug_path     = DEBUG_DIR     / f'{page_id}.json'

    if corrected_path.exists():
        n_skipped += 1
        continue
    if not conf_path.exists():
        continue

    annotated_text, n_low = conf_file_to_annotated(conf_path)

    if n_low == 0:
        if orig_path.exists():
            corrected_path.write_text(orig_path.read_text(encoding='utf-8'), encoding='utf-8')
        n_copied += 1
        continue

    corrected_text, suggestions_raw, suggestions_filtered, page_rejected, page_remaining = \
        postprocess_with_qwen(annotated_text, page_id)

    n_rejected  += page_rejected
    n_remaining += page_remaining

    corrected_path.write_text(corrected_text, encoding='utf-8')

    debug_data = {
        'page_id':                    page_id,
        'n_low_conf':                 n_low,
        'n_auto_rejected':            page_rejected,
        'n_markers_without_proposal': page_remaining,
        'step1_suggestions_raw':      suggestions_raw,
        'step1_suggestions_filtered': suggestions_filtered,
    }
    debug_path.write_text(json.dumps(debug_data, ensure_ascii=False, indent=2), encoding='utf-8')

    n_corrected += 1
    tqdm.write(
        f'{page_id}: {n_low} markiert  |  '
        f'{page_rejected} auto-rejected  |  '
        f'{page_remaining} ohne Vorschlag (unverändert)'
    )

print(f'\nFew-Shot Postkorrektur abgeschlossen.')
print(f'  Qwen korrigiert        : {n_corrected}')
print(f'  Direkt kopiert         : {n_copied}  (keine unsicheren Wörter)')
print(f'  Übersprungen           : {n_skipped}  (bereits vorhanden)')
print(f'  Auto-rejected (ges.)   : {n_rejected}  (Levenshtein-Filter)')
print(f'  Ohne Vorschlag (ges.)  : {n_remaining}  (Marker entfernt, Wort unverändert)')

Qwen Few-Shot Postkorrektur:   0%|          | 1/923 [00:10<2:42:22, 10.57s/it]

B1_P012: 7 markiert  |  3 auto-rejected  |  1 ohne Vorschlag (unverändert)


Qwen Few-Shot Postkorrektur:   0%|          | 2/923 [00:29<3:59:39, 15.61s/it]

B1_P014: 28 markiert  |  2 auto-rejected  |  2 ohne Vorschlag (unverändert)


Qwen Few-Shot Postkorrektur:   0%|          | 3/923 [00:40<3:27:05, 13.51s/it]

...

B6_P220: 47 markiert  |  2 auto-rejected  |  26 ohne Vorschlag (unverändert)


Qwen Few-Shot Postkorrektur: 100%|█████████▉| 922/923 [5:16:24<00:22, 22.26s/it]

B6_P221: 50 markiert  |  0 auto-rejected  |  16 ohne Vorschlag (unverändert)


Qwen Few-Shot Postkorrektur: 100%|██████████| 923/923 [5:16:26<00:00, 20.57s/it]

B6_P234: 8 markiert  |  0 auto-rejected  |  6 ohne Vorschlag (unverändert)

Few-Shot Postkorrektur abgeschlossen.
  Qwen korrigiert        : 918
  Direkt kopiert         : 5  (keine unsicheren Wörter)
  Übersprungen           : 0  (bereits vorhanden)
  Auto-rejected (ges.)   : 1218  (Levenshtein-Filter)
  Ohne Vorschlag (ges.)  : 10200  (Marker entfernt, Wort unverändert)


---
## Stufe 3 – Lexikon-Korrektur (Ground-Truth-Abgleich)

Fängt TrOCR-Fehler, die **über** dem Konfidenz-Threshold lagen und daher nie
markiert wurden — z.B. `Nationalist(0.77)` → `Naturalist`.

**Prinzip:** Jedes Wort im Output, das *nicht* im human-verified Lexikon steht,
wird gegen das Lexikon geprüft. Gibt es ein Lexikon-Wort in geringer
Levenshtein-Distanz (adaptiv zur Wortlänge, gleicher Anfangsbuchstabe),
wird es ersetzt. Wörter die bereits im Lexikon stehen werden **nie** angefasst.

| Wortlänge | max. Distanz |
|---|---|
| 5–7 | 1 |
| 8–10 | 2 |
| ≥ 11 | 3 |

Kurze Wörter (< 5) und Wörter mit Ziffern werden übersprungen (zu riskant).

In [9]:
def _lexicon_max_dist(length):
    if length < 5:
        return 0   # nicht anfassen
    if length < 8:
        return 1
    if length < 11:
        return 2
    return 3


def _is_inflection(a, b):
    """
    True wenn a nur eine Flexionsform von b ist (oder umgekehrt) —
    z.B. 'Resolutions'/'Resolution', 'reached'/'reach'.
    Solche Wörter sind kein OCR-Fehler und dürfen nicht ersetzt werden.
    """
    a, b = a.lower(), b.lower()
    if len(a) > len(b):
        a, b = b, a
    return b.startswith(a) and b[len(a):] in ('s', 'es', 'd', 'ed', 'ing')


def _lexicon_lookup(word):
    """
    Sucht das beste Lexikon-Wort für ein unbekanntes Wort.
    Gibt (replacement, dist) oder (None, None) zurück.

    Bedingungen: gleicher Anfangsbuchstabe (case-insensitive),
    Levenshtein-Distanz ≤ adaptiv nach Wortlänge, bei Gleichstand höchste Frequenz.
    Flexionsformen (Plural, Vergangenheitsform) werden nie ersetzt.
    """
    max_dist = _lexicon_max_dist(len(word))
    if max_dist == 0 or any(ch.isdigit() for ch in word):
        return None, None

    first = word[0].lower()
    best_word, best_dist, best_freq = None, max_dist + 1, 0

    for cand, freq in LEXICON.items():
        if cand[0].lower() != first:
            continue
        if abs(len(cand) - len(word)) > max_dist:
            continue
        if _is_inflection(word, cand):
            return None, None   # Wort ist Flexionsform eines Lexikon-Worts → korrekt
        d = _levenshtein(word, cand)
        if d < best_dist or (d == best_dist and freq > best_freq):
            best_word, best_dist, best_freq = cand, d, freq

    if best_word is not None and best_dist <= max_dist:
        return best_word, best_dist
    return None, None


def lexicon_correct_file(txt_path):
    """
    Korrigiert eine Datei in-place gegen das Lexikon.
    Gibt Liste der Änderungen [(original, replacement, dist)] zurück.
    """
    changes   = []
    lines_out = []

    for line in txt_path.read_text(encoding='utf-8').splitlines():
        if line.startswith('===') or not line.strip():
            lines_out.append(line)
            continue

        new_words = []
        for w in line.split():
            core = _word_core(w)
            # Bekanntes Wort (case-insensitive) → nie anfassen
            if not core or core.lower() in LEXICON_LOWER:
                new_words.append(w)
                continue
            repl, dist = _lexicon_lookup(core)
            if repl is None:
                new_words.append(w)
            else:
                # Interpunktion um das Wort erhalten
                new_words.append(w.replace(core, repl))
                changes.append((core, repl, dist))
        lines_out.append(' '.join(new_words))

    if changes:
        txt_path.write_text('\n'.join(lines_out), encoding='utf-8')
    return changes


# --- Auf alle korrigierten Dateien anwenden ---
from tqdm import tqdm

all_changes = []
n_files_changed = 0

files = sorted(CORRECTED_DIR.glob('*.txt'))
for txt_path in tqdm(files, desc='Lexikon-Korrektur'):
    changes = lexicon_correct_file(txt_path)
    if changes:
        n_files_changed += 1
        all_changes.extend(changes)

print(f'\nLexikon-Korrektur abgeschlossen.')
print(f'  Dateien geprüft   : {len(files)}')
print(f'  Dateien verändert : {n_files_changed}')
print(f'  Korrekturen gesamt: {len(all_changes)}')

if all_changes:
    print('\nHäufigste Korrekturen:')
    corr_counter = Counter((o, r) for o, r, _ in all_changes)
    for (orig, repl), cnt in corr_counter.most_common(25):
        print(f'  {orig!r:20s} → {repl!r:20s}  ({cnt}×)')

Lexikon-Korrektur: 100%|██████████| 923/923 [00:29<00:00, 31.17it/s]


Lexikon-Korrektur abgeschlossen.
  Dateien geprüft   : 923
  Dateien verändert : 903
  Korrekturen gesamt: 6104

Häufigste Korrekturen:
  'Zealand'            → 'Zeeland'             (143×)
  'March'              → 'Marsh'               (98×)
  'Ocean'              → 'Oceano'              (49×)
  'sleep'              → 'Sleet'               (48×)
  'wings'              → 'winds'               (46×)
  'given'              → 'Giver'               (38×)
  'shoot'              → 'short'               (38×)
  'showed'             → 'Shower'              (35×)
  'offered'            → 'offerred'            (31×)
  'words'              → 'wards'               (31×)
  'party'              → 'part'                (29×)
  'woman'              → 'women'               (28×)
  'struck'             → 'stuck'               (28×)
  'larger'             → 'large'               (28×)
  'peace'              → 'place'               (26×)
  'European'           → 'Europe'              (25×)
  'bread'     

---
## Stufe 4 – Globale Konsistenz-Normalisierung (Majority Voting)

Scannt alle korrigierten Seiten und findet Wörter/Bigramme die wahrscheinlich
OCR-Varianten desselben Tokens sind — z.B. `"Capd Cook"` (2×) vs `"Capt Cook"` (12×).

**Kriterien für eine Korrektur:**
- Levenshtein-Distanz ≤ 2 (Wort) bzw. ≤ 2 gesamt (Bigramm)
- Die Minderheitsform kommt ≤ 1/`MIN_RATIO` so oft vor wie die Mehrheitsform
- Die Mehrheitsform kommt mindestens `MIN_CANONICAL` mal vor

Läuft **nach** der Lexikon-Korrektur — normalisiert Varianten, die weder markiert
wurden noch im Lexikon stehen. Alle Dateien werden in-place aktualisiert.

In [10]:
from collections import Counter

# --- Parameter ---
MAJORITY_MAX_DIST      = 2   # max. Levenshtein-Distanz zwischen Variante und Canonical
MAJORITY_MIN_RATIO     = 4   # Mehrheitsform muss ≥ 4× häufiger sein als Minderheitsform
MAJORITY_MIN_CANONICAL = 3   # Mehrheitsform muss mindestens 3× vorkommen


def _collect_frequencies(corrected_dir):
    """Zählt Wort- und Bigramm-Häufigkeiten über alle korrigierten Dateien."""
    word_counts   = Counter()
    bigram_counts = Counter()
    for txt_path in sorted(corrected_dir.glob('*.txt')):
        for line in txt_path.read_text(encoding='utf-8').splitlines():
            if line.startswith('===') or not line.strip():
                continue
            words = line.split()
            word_counts.update(words)
            bigram_counts.update(zip(words, words[1:]))
    return word_counts, bigram_counts


def _build_majority_dict(counts, max_dist, min_ratio, min_canonical, is_bigram=False):
    """
    Findet Minderheitsvarianten und ordnet sie der häufigsten ähnlichen Form zu.
    Gibt {variant → canonical} zurück.

    Sortierung nach Häufigkeit (häufigste = Canonical) stellt sicher,
    dass "Capt Cook" (12×) nicht zur Variante von "Capd Cook" (2×) wird.
    """
    items       = sorted(counts.keys(), key=lambda x: -counts[x])
    corrections = {}
    assigned    = set()

    for canonical in items:
        if canonical in assigned:
            continue
        c_count = counts[canonical]
        if c_count < min_canonical:
            break  # Rest hat noch niedrigere Frequenz
        assigned.add(canonical)

        for variant in items:
            if variant == canonical or variant in assigned:
                continue
            v_count = counts[variant]
            if v_count * min_ratio >= c_count:
                continue  # Zu häufig → keine Minderheitsform

            if is_bigram:
                dist = (_levenshtein(canonical[0], variant[0])
                        + _levenshtein(canonical[1], variant[1]))
            else:
                if abs(len(canonical) - len(variant)) > max_dist:
                    continue  # Schnell-Filter: Längenunterschied zu groß
                dist = _levenshtein(canonical, variant)

            if dist <= max_dist:
                corrections[variant] = canonical
                assigned.add(variant)

    return corrections


def _apply_global_corrections(corrected_dir, word_corr, bigram_corr):
    """Wendet globale Korrekturen in-place auf alle Dateien an."""
    n_files    = 0
    n_words    = 0
    n_bigrams  = 0

    for txt_path in sorted(corrected_dir.glob('*.txt')):
        original  = txt_path.read_text(encoding='utf-8')
        lines_out = []
        changed   = False

        for line in original.splitlines():
            if line.startswith('===') or not line.strip():
                lines_out.append(line)
                continue

            words     = line.split()
            new_words = []
            i         = 0
            while i < len(words):
                # Bigramm zuerst prüfen
                if i + 1 < len(words):
                    bigram = (words[i], words[i + 1])
                    if bigram in bigram_corr:
                        w1, w2 = bigram_corr[bigram]
                        new_words += [w1, w2]
                        i        += 2
                        n_bigrams += 1
                        changed   = True
                        continue
                # Dann Einzelwort
                if words[i] in word_corr:
                    new_words.append(word_corr[words[i]])
                    n_words += 1
                    changed  = True
                else:
                    new_words.append(words[i])
                i += 1

            lines_out.append(' '.join(new_words))

        if changed:
            txt_path.write_text('\n'.join(lines_out), encoding='utf-8')
            n_files += 1

    return n_files, n_words, n_bigrams


print('Majority-Voting Funktionen definiert.')

Majority-Voting Funktionen definiert.


In [11]:
print('Häufigkeiten sammeln...')
word_counts, bigram_counts = _collect_frequencies(CORRECTED_DIR)
print(f'  Einzigartige Wörter  : {len(word_counts):,}')
print(f'  Einzigartige Bigramme: {len(bigram_counts):,}')

print('\nMajority-Dicts aufbauen...')
word_corr   = _build_majority_dict(
    word_counts,
    max_dist=MAJORITY_MAX_DIST, min_ratio=MAJORITY_MIN_RATIO,
    min_canonical=MAJORITY_MIN_CANONICAL, is_bigram=False,
)
bigram_corr = _build_majority_dict(
    bigram_counts,
    max_dist=MAJORITY_MAX_DIST, min_ratio=MAJORITY_MIN_RATIO,
    min_canonical=MAJORITY_MIN_CANONICAL, is_bigram=True,
)
print(f'  Wortkorrekturen      : {len(word_corr)}')
print(f'  Bigramm-Korrekturen  : {len(bigram_corr)}')

# Vorschau der gefundenen Korrekturen (Top 30 nach Häufigkeit der Variante)
print('\nTop Wortkorrekturen (Variante → Canonical  |  Häufigkeiten):')
top_word = sorted(word_corr.items(), key=lambda x: -word_counts[x[0]])[:30]
for variant, canonical in top_word:
    print(f'  {variant!r:20s} → {canonical!r:20s}  '
          f'({word_counts[variant]}× → {word_counts[canonical]}×)')

print('\nTop Bigramm-Korrekturen:')
top_bigram = sorted(bigram_corr.items(), key=lambda x: -bigram_counts[x[0]])[:20]
for variant, canonical in top_bigram:
    v_str = f'"{variant[0]} {variant[1]}"'
    c_str = f'"{canonical[0]} {canonical[1]}"'
    print(f'  {v_str:30s} → {c_str:30s}  '
          f'({bigram_counts[variant]}× → {bigram_counts[canonical]}×)')

Häufigkeiten sammeln...
  Einzigartige Wörter  : 30,546
  Einzigartige Bigramme: 114,809

Majority-Dicts aufbauen...
  Wortkorrekturen      : 16189
  Bigramm-Korrekturen  : 30184

Top Wortkorrekturen (Variante → Canonical  |  Häufigkeiten):
  'we'                 → 'the'                 (2980× → 14960×)
  'The'                → 'the'                 (2838× → 14960×)
  'at'                 → '&'                   (2250× → 11074×)
  'on'                 → '&'                   (1981× → 11074×)
  'I'                  → '&'                   (1620× → 11074×)
  'is'                 → '&'                   (1594× → 11074×)
  'it'                 → '&'                   (1587× → 11074×)
  'that'               → 'the'                 (1556× → 14960×)
  'had'                → 'a'                   (1450× → 6230×)
  'for'                → 'of'                  (1418× → 6847×)
  'they'               → 'the'                 (1341× → 14960×)
  'We'                 → 'the'                 (1276× → 1

In [12]:
# Vorschau prüfen — dann ausführen
# Falls Korrekturen unerwünscht sind: MAJORITY_MIN_RATIO oder MAJORITY_MIN_CANONICAL erhöhen

print('Globale Korrekturen anwenden...')
n_files, n_words_fixed, n_bigrams_fixed = _apply_global_corrections(
    CORRECTED_DIR, word_corr, bigram_corr
)

print(f'  Dateien verändert    : {n_files}')
print(f'  Wort-Korrekturen     : {n_words_fixed}')
print(f'  Bigramm-Korrekturen  : {n_bigrams_fixed}')
print('\nGlobale Konsistenz-Normalisierung abgeschlossen.')

Globale Korrekturen anwenden...
  Dateien verändert    : 923
  Wort-Korrekturen     : 61288
  Bigramm-Korrekturen  : 44517

Globale Konsistenz-Normalisierung abgeschlossen.


---
## Gesamtdokument zusammenstellen

In [13]:
total_lines = 0
missing     = []

with open(CORR_DOC_PATH, 'w', encoding='utf-8') as out:
    for page_entry in pages:
        page_id  = page_entry['page_id']
        txt_path = CORRECTED_DIR / f'{page_id}.txt'
        if not txt_path.exists():
            missing.append(page_id)
            continue
        content = txt_path.read_text(encoding='utf-8').strip()
        out.write(content + '\n\n')
        total_lines += len([l for l in content.splitlines() if not l.startswith('===')])

print(f'Ausgabe: {CORR_DOC_PATH}')
print(f'Seiten  : {len(pages) - len(missing)} / {len(pages)}')
print(f'Zeilen  : {total_lines}')
if missing:
    print(f'Fehlend : {len(missing)}')

Ausgabe: /home/justin/Ginger_Gradient/14/project/Capstone-Project/data/corrected_raw_document_fewshot.txt
Seiten  : 923 / 923
Zeilen  : 29883
